# ANON4REID - RAD Dataset Generation (2x T4 Kaggle)
This notebook generates the Level 3 (RAD - Stable Diffusion + ControlNet OpenPose + LCM-LoRA) dataset. It is optimized to run on a **Kaggle Dual T4 GPU** instance by splitting the dataset into two halves and spawning a dedicated worker process for each GPU (`cuda:0` and `cuda:1`).

In [ ]:
!pip install diffusers transformers accelerate controlnet_aux opencv-python Pillow numpy peft xformers

In [ ]:
# Write the worker logic and pipeline into a standalone py file.
# This prevents the multiprocessing wrapper from failing to find '__main__' methods in Jupyter notebooks.
worker_script = """
import os
import math
import shutil
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.multiprocessing as mp
from PIL import Image
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, LCMScheduler
from controlnet_aux import OpenposeDetector

ORIGINAL_SIZE = (64, 128)
MODEL_SIZE    = (256, 512)
BATCH_SIZE    = 4  # Process multiple images simultaneously to max out the T4 usage

def preprocess(image: Image.Image) -> Image.Image:
    image = image.convert("RGB")
    return image.resize(MODEL_SIZE, Image.LANCZOS)

def postprocess(image: Image.Image) -> Image.Image:
    return image.resize(ORIGINAL_SIZE, Image.LANCZOS)

def load_pipeline(device_id: int):
    device = f"cuda:{device_id}"
    print(f"[{device}] Loading ControlNet...")
    controlnet = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_openpose",
        torch_dtype=torch.float16,
    )
    
    print(f"[{device}] Loading Stable Diffusion v1.5...")
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "stable-diffusion-v1-5/stable-diffusion-v1-5",
        controlnet=controlnet,
        torch_dtype=torch.float16,
        safety_checker=None,
    ).to(device)

    print(f"[{device}] Injecting LCM-LoRA...")
    pipe.load_lora_weights("latent-consistency/lcm-lora-sdv1-5")
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config)
    
    # Use memory efficient attention via xformers (Massive speed boost over attention slicing)
    try:
        pipe.enable_xformers_memory_efficient_attention()
    except Exception as e:
        print(f"[{device}] Could not enable xformers: {e}")
        
    pipe.set_progress_bar_config(disable=True)  # Disable diffusers internal pbar
    
    print(f"[{device}] Loading OpenPose Detector...")
    detector = OpenposeDetector.from_pretrained("lllyasviel/ControlNet").to(device)
    
    return pipe, detector

def gpu_worker(gpu_id: int, image_paths: list):
    print(f"--> Worker on cuda:{gpu_id} started. Assigned {len(image_paths)} images.")
    
    pipe, detector = load_pipeline(gpu_id)
    
    # Pre-filter existing logic to avoid batching complications
    tasks = [(in_path, out_path) for in_path, out_path in image_paths if not os.path.exists(out_path)]
    skipped = len(image_paths) - len(tasks)
    
    print(f"[cuda:{gpu_id}] Skipped {skipped} already processed images.")
    
    processed = 0
    errors = 0
    milestone = max(1, len(tasks) // 10)
    
    # BATCH LOOP
    for i in range(0, len(tasks), BATCH_SIZE):
        batch_tasks = tasks[i:min(i + BATCH_SIZE, len(tasks))]
        
        try:
            # 1. Load and slice
            imgs = [Image.open(t[0]).convert("RGB") for t in batch_tasks]
            imgs_up = [preprocess(img) for img in imgs]
            
            # 2. Extract poses 
            poses = [detector(img) for img in imgs_up]
            
            # 3. Batch SD Inference
            prompts = ["a person, different appearance, full body, high quality, realistic"] * len(batch_tasks)
            neg_prompts = ["blurry, low quality, distorted, noise, same person"] * len(batch_tasks)
            
            results = pipe(
                prompt=prompts,
                negative_prompt=neg_prompts,
                image=poses,
                height=512,
                width=256,
                num_inference_steps=8,
                guidance_scale=1.5,
                controlnet_conditioning_scale=0.8,
            ).images
            
            # 4. Save results
            for j, res in enumerate(results):
                res_down = postprocess(res)
                res_down.save(batch_tasks[j][1])
            
            processed += len(batch_tasks)
            
        except Exception as e:
            print(f"[cuda:{gpu_id}] Error on batch starting at {batch_tasks[0][0]}: {e}")
            errors += len(batch_tasks)

        if processed > 0 and processed % milestone < BATCH_SIZE:
            print(f"[cuda:{gpu_id}] Progress: {processed}/{len(tasks)}")

    print(f"<-- Worker on cuda:{gpu_id} finished. Processed: {processed}, Errors: {errors}")

if __name__ == '__main__':
    try:
        mp.set_start_method('spawn', force=True)
    except RuntimeError:
        pass

    INPUT_DIR = "/kaggle/input/market-1501"   # Adjust to match Kaggle input
    OUTPUT_DIR = "/kaggle/working/Market-1501-RAD"
    SPLITS = ["query", "bounding_box_test", "bounding_box_train"]

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for split in SPLITS:
        os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

    all_tasks = []
    for split in SPLITS:
        split_dir_in = os.path.join(INPUT_DIR, split)
        split_dir_out = os.path.join(OUTPUT_DIR, split)
        if os.path.exists(split_dir_in):
            for fname in os.listdir(split_dir_in):
                if fname.endswith('.jpg'):
                    all_tasks.append(
                        (os.path.join(split_dir_in, fname), os.path.join(split_dir_out, fname))
                    )
                    
    total_images = len(all_tasks)
    print(f"Total images to process: {total_images}")
    
    if total_images > 0:
        midpoint = math.ceil(total_images / 2)
        tasks_gpu0 = all_tasks[:midpoint]
        tasks_gpu1 = all_tasks[midpoint:]
        
        p0 = mp.Process(target=gpu_worker, args=(0, tasks_gpu0))
        p1 = mp.Process(target=gpu_worker, args=(1, tasks_gpu1))
        
        p0.start()
        p1.start()
        
        p0.join()
        p1.join()
        
        print("Generation Complete across both GPUs!")
    else:
        print("No images found. Check INPUT_DIR.")
"""

with open("rad_worker.py", "w") as f:
    f.write(worker_script)

print("Generated rad_worker.py successfully!")

In [ ]:
!python rad_worker.py

In [ ]:
import shutil

OUTPUT_DIR = "/kaggle/working/Market-1501-RAD"

# --- Zipping ---
print("Zipping Market-1501-RAD...")
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f"Zipping Complete! You can now download {OUTPUT_DIR}.zip")